# PySpark MLlib – Linear Regression
## Boston Housing Dataset

**Objective:** Predict median house value (`medv`) from neighborhood characteristics using PySpark MLlib.

- **Problem type:** Supervised regression
- **Algorithm:** Linear Regression
- **Target:** `medv`
- **Tools:** Python, Apache Spark, PySpark MLlib

**Business context:** A model can learn relationships between neighborhood characteristics and historical home values. This educational dataset should not be used for real-world property valuation or decisions affecting people. The original Boston Housing dataset has known ethical and fairness limitations.

## 1. Install PySpark

Run this cell in Google Colab. If Colab asks you to restart the runtime, follow its prompt and then continue from the top.

In [ ]:
!pip -q install pyspark

## 2. Import Libraries and Create SparkSession

`SparkSession` is the entry point for Spark DataFrames and MLlib. The `local[*]` setting lets Spark use the CPU cores available in the Colab runtime.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = (
    SparkSession.builder
    .appName("Boston Housing Regression")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)

## 3. Upload and Load the Dataset

Upload your `Boston.csv` file using the picker below. The CSV should contain the columns listed in the project description, including `medv`.

If you do not have the file yet, obtain a permitted copy of the dataset and upload it here.

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload Boston.csv to continue.")

# Use the first uploaded file; rename it to Boston.csv if necessary.
uploaded_filename = next(iter(uploaded))
print("Uploaded file:", uploaded_filename)

df = spark.read.csv(
    uploaded_filename,
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

print("Rows:", df.count())
print("Columns:", len(df.columns))

## 4. Select Features and Target

**Features** are input variables used for prediction. **Target/label** is the value the model learns to predict.

The `Unnamed: 0` column sometimes appears when a CSV was saved with a pandas index. It is only a row identifier, so we do not use it as a feature.

In [ ]:
feature_columns = [
    "crim", "zn", "indus", "chas", "nox",
    "rm", "age", "dis", "rad", "tax",
    "ptratio", "black", "lstat"
]
target_column = "medv"

# Make column matching more robust to accidental surrounding spaces.
df = df.toDF(*[c.strip() for c in df.columns])

required_columns = feature_columns + [target_column]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(
        f"Missing expected columns: {missing_columns}. "
        f"Available columns: {df.columns}"
    )

data = df.select(*required_columns)

data.show(5)
print("Selected columns:", data.columns)

## 5. Check Missing Values

Check for null values before building the feature vector. This notebook removes rows that have missing values in any selected feature or the target. For a larger project, consider fitting an imputation strategy on the training data instead.

In [ ]:
data.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in data.columns
]).show()

rows_before = data.count()
data = data.dropna(subset=required_columns)
rows_after = data.count()

print("Rows before removing missing values:", rows_before)
print("Rows after removing missing values:", rows_after)
print("Rows removed:", rows_before - rows_after)

## 6. Assemble Features into a Vector

Spark MLlib expects model inputs in a single vector column. `VectorAssembler` combines the 13 feature columns into `features`.

In [ ]:
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features",
    handleInvalid="error"
)

assembled_data = assembler.transform(data).select(
    "features",
    col(target_column).cast("double").alias(target_column)
)

assembled_data.show(5, truncate=False)

## 7. Split into Training and Testing Sets

Use 80% of the rows for training and 20% for testing. The fixed seed makes the split reproducible for the same dataset and runtime behavior.

In [ ]:
train_data, test_data = assembled_data.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training rows:", train_data.count())
print("Testing rows:", test_data.count())

## 8. Create and Train the Linear Regression Model

Linear Regression predicts a continuous numeric value by fitting a relationship between the input features and the target.

In [ ]:
lr = LinearRegression(
    featuresCol="features",
    labelCol=target_column,
    predictionCol="prediction",
    maxIter=100
)

model = lr.fit(train_data)
print("Model trained successfully!")

## 9. Generate Predictions

- `medv`: Actual target value in the dataset.
- `prediction`: Value estimated by the trained model.

In [ ]:
predictions = model.transform(test_data)

predictions.select(
    "medv",
    "prediction"
).show(10, truncate=False)

## 10. Evaluate the Model

- **RMSE (Root Mean Squared Error):** Measures prediction error in the target's units. Lower is generally better.
- **R² (coefficient of determination):** Compares the model with a mean-based baseline. Values closer to 1 generally indicate a stronger fit; a negative test-set value is possible.

Actual scores depend on the dataset and split.

In [ ]:
evaluator_rmse = RegressionEvaluator(
    labelCol=target_column,
    predictionCol="prediction",
    metricName="rmse"
)
evaluator_r2 = RegressionEvaluator(
    labelCol=target_column,
    predictionCol="prediction",
    metricName="r2"
)
evaluator_mae = RegressionEvaluator(
    labelCol=target_column,
    predictionCol="prediction",
    metricName="mae"
)

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)

print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")
print(f"MAE:  {mae:.4f}")

## 11. Inspect Model Coefficients

Each coefficient describes the model's estimated change in prediction for a one-unit increase in that feature, holding other features constant. This is a model association, not proof of causation.

In [ ]:
print(f"Intercept: {model.intercept:.4f}")
print("\nFeature coefficients:")

for feature, coefficient in zip(feature_columns, model.coefficients):
    print(f"{feature:>8}: {coefficient: .4f}")

## 12. Workflow Summary

**CSV → Spark DataFrame → Select Features and Target → Check Missing Values → VectorAssembler → Train/Test Split → Linear Regression → Predictions → Evaluation**

### Key terms

| Term | Meaning |
|---|---|
| Feature | Input variable used for prediction |
| Label/target | Value the model predicts (`medv`) |
| VectorAssembler | Combines input columns into one feature vector |
| Linear Regression | Algorithm for predicting continuous values |
| RMSE | Error measure; lower is generally better |
| MAE | Mean absolute prediction error |
| R² | Model fit relative to a mean-based baseline |

### Further experiments
1. Compare with Decision Tree Regression or Random Forest Regression.
2. Examine residuals to identify systematic errors.
3. Try regularization and feature scaling.
4. Compare models on the same test split.
5. Consider the dataset's historical and ethical limitations before interpreting results.

## 13. Stop Spark

Run this cell when you have finished working with Spark. If you want to continue using Spark later in the notebook, run this cell only at the end.

In [ ]:
spark.stop()
print("Spark session stopped.")